# Semana 5 · Evaluación final, análisis de errores y predicción de julio

## 0. Registro previo antes de abrir el test (2026)
- **Modelo seleccionado (Semana 3):** `LinearRegression` (menor MAE en validación: 13.62).
- **Parámetros:** Valores por defecto (`fit_intercept=True`), sin modificar para evitar sobreajuste.
- **Predictores admitidos:** `channel`, `product_line`, `region`, `stock_units`, `marketing_budget`, `planned_discount`, `list_price`, `campaign`, `days_in_month`, `año`, `numero_mes`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Cargamos los datos históricos ordenados
datos = pd.read_csv('../data/sales_history.csv')
datos = datos.sort_values(by=['channel', 'product_line', 'region', 'month']).copy()
print('Total de filas cargadas:', len(datos))
datos.head(3)

Total de filas cargadas: 1188


,month,channel,product_line,region,marketing_spend,avg_discount,stock_units,website_sessions,sales_units,revenue,marketing_budget,planned_discount,list_price,campaign,days_in_month,closing_stock_units,stockout
15,2021-01,ecommerce,electric,coast,2400.37,0.037,51,5119.0,41,58434.84,2588.44,0.045,1480.00,none,31,10,0
33,2021-02,ecommerce,electric,coast,2252.26,0.082,61,5569.0,41,55787.04,2166.71,0.067,1482.20,none,28,20,0
51,2021-03,ecommerce,electric,coast,2981.02,0.034,78,6276.0,70,100375.80,3020.43,0.038,1484.41,none,31,8,0


## 1. Preparación de datos y partición temporal
Quitamos las variables posteriores al mes (fuga de datos) y dividimos en **2 bloques**: Entrenamiento con todo el histórico disponible (`2022-01` a `2025-12`) y Test reservado (`2026-01` a `2026-06`).

In [2]:
# Columnas posteriores al inicio de mes (se descartan)
columnas_a_descartar = [
    'sales_units', 'closing_stock_units', 'revenue', 'stockout',
    'website_sessions', 'marketing_spend', 'avg_discount'
]

# Rellenamos nulos conocidos en predictores
datos['marketing_budget'] = datos['marketing_budget'].fillna(0)
datos['planned_discount'] = datos['planned_discount'].fillna(0)

# Extraemos año y mes numérico para el modelo
datos['fecha_mes'] = pd.to_datetime(datos['month'], format='%Y-%m')
datos['año'] = datos['fecha_mes'].dt.year
datos['numero_mes'] = datos['fecha_mes'].dt.month

# Matriz de variables numéricas (X) y ventas reales (y)
variables_predictoras = datos.drop(columns=columnas_a_descartar + ['month', 'fecha_mes'])
matriz_variables = pd.get_dummies(variables_predictoras, drop_first=False, dtype=int)
ventas_reales = datos['sales_units']

# Dos particiones claras: Entrenamiento (2022-2025) y Prueba (2026)
filtro_entrenamiento = datos['month'].between('2022-01', '2025-12')
filtro_prueba = datos['month'].between('2026-01', '2026-06')

X_entrenamiento, y_entrenamiento = matriz_variables[filtro_entrenamiento], ventas_reales[filtro_entrenamiento]
X_prueba, y_prueba = matriz_variables[filtro_prueba], ventas_reales[filtro_prueba]

print('Entrenamiento (2022-2025):', len(X_entrenamiento), 'filas')
print('Prueba (2026-01 a 06):    ', len(X_prueba), 'filas')

Entrenamiento (2022-2025): 864 filas
Prueba (2026-01 a 06):     108 filas


## 2. Evaluación única en Test (2026) y comparativa con Validación (Semana 3)
Entrenamos el modelo una sola vez con todo el histórico (`2022-2025`) y evaluamos en `2026`.

In [3]:
# Entrenamos LinearRegression una sola vez
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_entrenamiento, y_entrenamiento)

# Predicciones sobre el test reservado de 2026
predicciones_prueba = modelo_lineal.predict(X_prueba)

mae = mean_absolute_error(y_prueba, predicciones_prueba)
rmse = root_mean_squared_error(y_prueba, predicciones_prueba)
wape = 100 * np.sum(np.abs(y_prueba - predicciones_prueba)) / np.sum(y_prueba)
r2 = r2_score(y_prueba, predicciones_prueba)

# Comparamos el resultado de 2026 con el obtenido en Validación (2025) en la Semana 3
comparativa_metricas = pd.DataFrame([
    {'Conjunto': 'Validación 2025 (Semana 3)', 'MAE': 13.62, 'RMSE': 18.31, 'WAPE (%)': 13.12, 'R2': 0.85},
    {'Conjunto': 'Test reservado 2026', 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'WAPE (%)': round(wape, 2), 'R2': round(r2, 2)}
])
comparativa_metricas

,Conjunto,MAE,RMSE,WAPE (%),R2
0,Validación 2025 (Semana 3),13.62,18.31,13.12,0.85
1,Test reservado 2026,17.47,22.03,14.65,0.83


## 3. Análisis de casos de error en 2026
Buscamos las filas donde el modelo ha fallado más en 2026 para entender el motivo (roturas de stock, campañas o diferencias de grupo).

In [4]:
errores_2026 = datos.loc[filtro_prueba, ['month', 'channel', 'product_line', 'region', 'stock_units', 'stockout', 'campaign', 'sales_units']].copy()
errores_2026['prediccion'] = np.round(predicciones_prueba, 1)
errores_2026['error_unidades'] = np.round(np.abs(errores_2026['sales_units'] - errores_2026['prediccion']), 1)

# Mostramos los 3 mayores errores de 2026
errores_2026.sort_values('error_unidades', ascending=False).head(3)

,month,channel,product_line,region,stock_units,stockout,campaign,sales_units,prediccion,error_unidades
1145,2026-04,ecommerce,urban,metro,356,0,spring,328,246.5,81.5
1184,2026-06,ecommerce,mountain,metro,198,1,none,198,140.2,57.8
1172,2026-06,store,urban,metro,205,0,none,202,150.3,51.7


**Explicación de dos casos de error:**
1. **2026-04 (ecommerce · urban · metro):** Venta real 328 vs Predicción 246.5 (Error: 81.5). La combinación de campaña `spring` en zona metropolitana y canal online disparó la demanda muy por encima de lo que estima una recta lineal.
2. **2026-06 (ecommerce · mountain · metro):** Venta real 198 vs Predicción 140.2 (Error: 57.8). Aquí hubo **rotura de stock (`stockout = 1`)**: se vendió todo lo disponible (198 de 198) y la demanda real no servida fue incluso mayor.

## 4. Factores que influyen en el modelo (Permutación) y límites
Desordenamos cada columna en el test de 2026 para medir cuánto empeora el error (MAE) y saber qué variables mandan.

In [5]:
resultado_permutacion = permutation_importance(modelo_lineal, X_prueba, y_prueba, scoring='neg_mean_absolute_error', n_repeats=5, random_state=42)
importancia_variables = pd.Series(resultado_permutacion.importances_mean, index=X_entrenamiento.columns).sort_values(ascending=False)

tabla_cinco_importantes = importancia_variables.head(5).round(2).reset_index()
tabla_cinco_importantes.columns = ['Variable', 'Subida del error si se desordena (MAE)']
tabla_cinco_importantes

,Variable,Subida del error si se desordena (MAE)
0,list_price,64.11
1,product_line_electric,48.06
2,product_line_urban,29.65
3,stock_units,24.42
4,product_line_mountain,6.10


**Límites de la interpretación:**  
Los factores que más influyen en el modelo son el precio de tarifa (list_price), la línea de producto (electric, urban) y el stock inicial (stock_units). Sin embargo, al tratarse de datos sintéticos, esto muestra asociaciones estadísticas predictivas y **no demuestra causalidad real en el mercado** (cambiar el precio o el stock artificialmente no garantiza las mismas ventas reales).

## 5. Predicción de julio 2026 (18 combinaciones)
Creamos el DataFrame de julio con las 18 combinaciones y calculamos las predicciones estimadas.

In [6]:
# Creamos el DataFrame de julio usando las 18 combinaciones de junio
datos_julio = datos[datos['month'] == '2026-06'].copy()
datos_julio['month'] = '2026-07'
datos_julio['numero_mes'] = 7
datos_julio['days_in_month'] = 31

# Preparamos las columnas igual que en el entrenamiento
X_julio = pd.get_dummies(datos_julio.drop(columns=columnas_a_descartar + ['month', 'fecha_mes']), drop_first=False, dtype=int)
X_julio = X_julio.reindex(columns=X_entrenamiento.columns, fill_value=0)

# DataFrame final con las 18 predicciones y guardado en CSV
df_predicciones_julio = datos_julio[['month', 'channel', 'product_line', 'region']].copy()
df_predicciones_julio['prediccion'] = np.round(modelo_lineal.predict(X_julio), 1)
df_predicciones_julio.to_csv('../evidencies/predicciones_julio_2026.csv', index=False)
df_predicciones_julio

,month,channel,product_line,region,prediccion
1185,2026-07,ecommerce,electric,coast,102.5
1186,2026-07,ecommerce,electric,inland,62.8
1187,2026-07,ecommerce,electric,metro,121.0
1182,2026-07,ecommerce,mountain,coast,132.2
1183,2026-07,ecommerce,mountain,inland,81.6
1184,2026-07,ecommerce,mountain,metro,141.6
1179,2026-07,ecommerce,urban,coast,198.7
1180,2026-07,ecommerce,urban,inland,125.5
1181,2026-07,ecommerce,urban,metro,208.0
1176,2026-07,store,electric,coast,77.5


## 6. Recomendaciones de negocio (Stock y Campaña)
- **Recomendación de stock (Confianza alta):** Aumentar el stock de seguridad en las combinaciones de verano con más demanda (`ecommerce` y `metro` en líneas `urban` y `mountain`), ya que en junio ya sufrimos roturas de stock (`stockout = 1`) y es el factor que más limita las ventas.
- **Recomendación de campaña (Confianza media-baja):** Mantener campañas en primavera/verano, pero sin esperar milagros solo por subir el presupuesto publicitario (`marketing_budget` tiene un impacto pequeño en el modelo lineal) si antes no se garantiza que haya bicicletas disponibles en el almacén.